In [ ]:
import os
import json
from clickhouse_driver import Client
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, lit, sha2
from pyspark.sql.types import BooleanType, DateType, DecimalType, DoubleType, IntegerType, LongType, StringType, StructField, StructType, TimestampType

def env_value(key):
    value = os.environ.get(key)
    if value is None or value == "":
        raise ValueError(f"Missing environment variable: {key}")
    return value

def load_json_env(key):
    return json.loads(env_value(key))

def build_schema(definition):
    mapping = {
        "string": StringType(),
        "int": IntegerType(),
        "long": LongType(),
        "double": DoubleType(),
        "decimal": DecimalType(38, 12),
        "date": DateType(),
        "timestamp": TimestampType(),
        "boolean": BooleanType()
    }
    fields = []
    for column in definition:
        column_type = column["type"].lower()
        if column_type.startswith("decimal("):
            precision, scale = column_type.replace("decimal(", "").replace(")", "").split(",")
            dtype = DecimalType(int(precision), int(scale))
        else:
            dtype = mapping[column_type]
        fields.append(StructField(column["name"], dtype, column.get("nullable", False)))
    return StructType(fields)

def spark_session():
    return (
        SparkSession.builder.appName("oracle_to_clickhouse_staging")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.sql.shuffle.partitions", env_value("SPARK_SHUFFLE_PARTITIONS"))
        .config("spark.sql.autoBroadcastJoinThreshold", env_value("SPARK_BROADCAST_THRESHOLD"))
        .config("spark.sql.session.timeZone", "UTC")
        .getOrCreate()
    )

def clickhouse_type(spark_type):
    if isinstance(spark_type, StringType):
        return "String"
    if isinstance(spark_type, IntegerType):
        return "Int32"
    if isinstance(spark_type, LongType):
        return "Int64"
    if isinstance(spark_type, DoubleType):
        return "Float64"
    if isinstance(spark_type, DecimalType):
        return f"Decimal({spark_type.precision},{spark_type.scale})"
    if isinstance(spark_type, TimestampType):
        return "DateTime64(3)"
    if isinstance(spark_type, DateType):
        return "Date"
    if isinstance(spark_type, BooleanType):
        return "UInt8"
    raise TypeError(f"Unsupported type: {spark_type}")

def create_clickhouse_table(schema):
    database = env_value("CLICKHOUSE_DATABASE")
    table = env_value("CLICKHOUSE_STAGING_TABLE")
    order_by = load_json_env("CLICKHOUSE_ORDER_BY")
    client = Client(
        host=env_value("CLICKHOUSE_HOST"),
        port=int(env_value("CLICKHOUSE_PORT")),
        user=env_value("CLICKHOUSE_USER"),
        password=env_value("CLICKHOUSE_PASSWORD"),
        secure=env_value("CLICKHOUSE_SECURE").lower() == "true",
        database=database
    )
    columns = []
    for field in schema.fields:
        columns.append(f"{field.name} {clickhouse_type(field.dataType)}")
    columns.append("snapshot_date Date")
    columns.append("record_hash FixedString(64)")
    ddl = f"CREATE TABLE IF NOT EXISTS {database}.{table} (" + " ,".join(columns).replace(" ,", ",") + " ) ENGINE = MergeTree PARTITION BY snapshot_date ORDER BY (" + ",".join(order_by) + ") SETTINGS index_granularity = 8192"
    client.execute(ddl)

source_schema_def = load_json_env("ORACLE_SCHEMA_JSON")
source_schema = build_schema(source_schema_def)
create_clickhouse_table(source_schema)
spark = spark_session()


In [ ]:
oracle_query = env_value("ORACLE_SOURCE_QUERY")
partition_column = env_value("ORACLE_PARTITION_COLUMN")
lower_bound = env_value("ORACLE_LOWER_BOUND")
upper_bound = env_value("ORACLE_UPPER_BOUND")
num_partitions = env_value("ORACLE_NUM_PARTITIONS")
snapshot_date = env_value("SNAPSHOT_DATE")
base_df = (
    spark.read.schema(source_schema)
    .format("jdbc")
    .option("url", env_value("ORACLE_JDBC_URL"))
    .option("user", env_value("ORACLE_USER"))
    .option("password", env_value("ORACLE_PASSWORD"))
    .option("dbtable", f"({oracle_query}) src")
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("fetchsize", env_value("ORACLE_FETCH_SIZE"))
    .option("partitionColumn", partition_column)
    .option("lowerBound", lower_bound)
    .option("upperBound", upper_bound)
    .option("numPartitions", num_partitions)
    .load()
)
fields_for_hash = [field.name for field in source_schema]
transformed_df = base_df.withColumn("snapshot_date", lit(snapshot_date)).withColumn(
    "record_hash",
    sha2(concat_ws("||", *[col(column).cast("string") for column in fields_for_hash]), 256)
)


In [ ]:
clickhouse_jdbc_url = env_value("CLICKHOUSE_JDBC_URL")
staging_table = f"{env_value('CLICKHOUSE_DATABASE')}.{env_value('CLICKHOUSE_STAGING_TABLE')}"
(
    transformed_df.write.format("jdbc")
    .option("url", clickhouse_jdbc_url)
    .option("dbtable", staging_table)
    .option("user", env_value("CLICKHOUSE_USER"))
    .option("password", env_value("CLICKHOUSE_PASSWORD"))
    .option("driver", "ru.yandex.clickhouse.ClickHouseDriver")
    .option("batchsize", env_value("CLICKHOUSE_BATCH_SIZE"))
    .option("numPartitions", env_value("CLICKHOUSE_WRITE_PARTITIONS"))
    .mode("append")
    .save()
)
